# Stage B — UNet fine-tune (DPED iPhone → DSLR)

Prereqs:
1. Stage A checkpoint from a previous session (see stage_a notebook)
2. Add it as **Add Input → Your Work → Stage A output** (path: `/kaggle/input/<stage-a-slug>/runs/stage_a/...`)
3. Attach the Stage A data output too (`latent_cache`) to skip re-prep

In [ ]:
# Cell 1: install deps (torch/torchvision untouched — Kaggle's preinstalled matched pair is used)
!pip -q install diffusers==0.31.0 transformers==4.44.2 accelerate safetensors bitsandbytes scipy kagglehub tqdm pillow
import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
# Cell 2: get project code
# ROUTE 1 (easiest): Kaggle.com -> Datasets -> New Dataset -> upload dped-ldm-upload.zip (Kaggle auto-extracts),
#   then in this notebook: right panel -> Add Input -> Your Datasets -> attach it. REPO_URL stays empty.
# ROUTE 2: put code on GitHub and paste the URL below.
import os, glob, shutil, subprocess, sys
REPO_URL = ""  # e.g. "https://github.com/YOUR_USER/dped-ldm.git" (Route 2 only)
W = '/kaggle/working'
os.chdir(W)
if not os.path.exists(f'{W}/src/common.py'):
    hits = glob.glob('/kaggle/input/*/src/common.py')
    if hits:
        root = os.path.dirname(os.path.dirname(hits[0]))
        for item in ('src', 'tests', 'requirements.txt', 'README.md'):
            s, d = os.path.join(root, item), os.path.join(W, item)
            if os.path.isdir(s):
                shutil.copytree(s, d, dirs_exist_ok=True)
            elif os.path.exists(s):
                shutil.copy2(s, d)
        print('Code loaded from attached Kaggle dataset.')
    elif REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, f'{W}/repo'], check=True)
        shutil.copytree(f'{W}/repo/src', f'{W}/src', dirs_exist_ok=True)
        print('Code cloned from GitHub.')
    else:
        raise RuntimeError('Code not found: attach code dataset (Route 1) or set REPO_URL (Route 2)')
sys.path.insert(0, f'{W}/src')
print('src files:', sorted(os.listdir(f'{W}/src')))

In [ ]:
# Cell 3: pull ALL Stage A artifacts from attached inputs (latents, crops, test images) -- no recompute
import glob, shutil, os
from pathlib import Path
SRC = sorted(glob.glob('/kaggle/input/*'))
print('attached inputs:', SRC)

def find_one(pattern):
    for base in SRC:
        hit = glob.glob(f'{base}/**/{pattern}', recursive=True)
        if hit:
            return Path(hit[0])
    return None

dest_root = Path('data/processed/iphone')

cache_src = find_one('latent_cache/train_phone.pt')
assert cache_src, 'latent_cache/train_phone.pt not found in attached inputs (attach your dped-stage-a output dataset)'
cache_src = cache_src.parent
(dest_root / 'latent_cache').mkdir(parents=True, exist_ok=True)
for f in glob.glob(f'{cache_src}/*.pt'):
    shutil.copy(f, dest_root / 'latent_cache' / Path(f).name)
print('latent cache copied:', len(os.listdir(dest_root / 'latent_cache')), 'files')

crops_src = find_one('crops/val/phone')
if crops_src:
    crops_root = crops_src.parent.parent
    for split in ('train', 'val'):
        for sub in ('phone', 'dslr'):
            src = crops_root / split / sub
            dst = dest_root / 'crops' / split / sub
            dst.mkdir(parents=True, exist_ok=True)
            if src.exists() and not any(dst.iterdir()):
                for f in glob.glob(str(src / '*')):
                    shutil.copy(f, dst / Path(f).name)
    print('crops copied')
else:
    print('NOTE: crops not in attached inputs (only needed for val metrics)')

test_src = find_one('test/phone')
if test_src:
    for split in ('phone', 'dslr'):
        td = dest_root / 'test' / split
        td.mkdir(parents=True, exist_ok=True)
        for f in glob.glob(str(test_src.parent / split / '*')):
            shutil.copy(f, td / Path(f).name)
    print('test images copied:', len(os.listdir(dest_root / 'test' / 'phone')))
else:
    print('WARNING: test images not found in attached inputs; Cell 6 full-res will fail.')


In [ ]:
# Cell 4: Stage A checkpoint path (from attached input)
import glob
CKPT = sorted(glob.glob('/kaggle/input/**/controlnet_step_*.pt', recursive=True))[-1]
print('using ControlNet checkpoint:', CKPT)

In [ ]:
# Cell 5: Stage B fine-tune (~4-6h for 6k steps at effective batch 16 on T4)
import sys, glob, shutil, os; sys.path.insert(0, '/kaggle/working/src')
# resume bootstrap: fresh session me runs/ khali hota hai -> last checkpoint attached inputs se wapas
_cks = sorted(glob.glob('/kaggle/input/**/runs/stage_b/finetune_step_*.pt', recursive=True))
if _cks:
    _src = _cks[-1]
    _dst = os.path.join('runs/stage_b', os.path.basename(_src))
    if not os.path.exists(_dst):
        os.makedirs('runs/stage_b', exist_ok=True)
        shutil.copy(_src, _dst)
        print('resume bootstrap: restored', _dst)
sys.argv = ['finetune_unet.py',
            '--data', 'data/processed/iphone',
            '--out', 'runs/stage_b',
            '--init-controlnet', CKPT,
            '--steps', '6000', '--batch', '8', '--accum', '2',
            '--resume']
import finetune_unet
finetune_unet.main()

In [ ]:
# Cell 6: export diffusers format + full-res test inference with EMA weights
import sys, glob; sys.path.insert(0, '/kaggle/working/src')
sys.argv = ['infer.py', '--ckpt', sorted(glob.glob('runs/stage_b/finetune_step_*.pt'))[-1],
            '--mode', 'full', '--ema', '--num', '29', '--export', 'runs/export']
import infer; infer.main()